# YOLO26 车辆检测 - v2 优化版
imgsz=640 + 全层训练 + 早停

In [2]:
# v2 优化版训练
from ultralytics import YOLO
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

model = YOLO('C:/Users/ql157/yolo26n.pt')

results = model.train(
    data='E:/dev/yjwlYOLO/dev/dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    freeze=0,
    patience=7,
    project='E:/dev/yjwlYOLO/dev/runs',
    name='train_v2',
    exist_ok=True,
)

New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.13.5 torch-2.8.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:/dev/yjwlYOLO/dev/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:/Users/ql157/yolo26n.pt, momentum=0.937

KeyboardInterrupt: 

In [ ]:
# 评估v2模型
from ultralytics import YOLO

model = YOLO('E:/dev/yjwlYOLO/dev/runs/train_v2/weights/best.pt')

metrics = model.val(
    data='E:/dev/yjwlYOLO/dev/dataset/data.yaml',
    device=0,
    split='val'
)

print('='*50)
print('v2 模型评估结果')
print('='*50)
print(f'mAP@0.5:     {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:   {metrics.box.mp:.4f}')
print(f'Recall:      {metrics.box.mr:.4f}')
print('='*50)

print('\n各类别 AP@0.5:')
all_names = list(model.names.values())
for i, cls_idx in enumerate(metrics.box.ap_class_index):
    print(f'  {all_names[cls_idx]}: {metrics.box.ap50[i]:.4f}')

In [ ]:
# v1 vs v2 对比
import os
from ultralytics import YOLO

models = {
    'v1 (imgsz=480, freeze=10)': 'E:/dev/yjwlYOLO/dev/runs/train/weights/best.pt',
    'v2 (imgsz=640, freeze=0)':  'E:/dev/yjwlYOLO/dev/runs/train_v2/weights/best.pt',
}

print('='*60)
print('YOLO26 模型对比')
print('='*60)
for name, path in models.items():
    if os.path.exists(path):
        m = YOLO(path)
        r = m.val(data='E:/dev/yjwlYOLO/dev/dataset/data.yaml', device=0, split='val', verbose=False)
        print(f'{name}:')
        print(f'  mAP50={r.box.map50:.4f}  mAP50-95={r.box.map:.4f}  P={r.box.mp:.4f}  R={r.box.mr:.4f}')
    else:
        print(f'{name}: 未训练')
print('='*60)

测试5060那边的训练效果怎么样 和本地4060的训练效果对比

In [1]:
from ultralytics import YOLO

model = YOLO('E:/dev/yjwlYOLO/dev/runs/train_v2/weights/5060ti1.pt')
results = model.val(data='E:/dev/yjwlYOLO/dev/dataset/data.yaml', device='cpu')

print(f"mAP50:    {results.box.map50:.4f}")
print(f"mAP50-95: {results.box.map:.4f}")
print(f"Precision: {results.box.mp:.4f}")
print(f"Recall:    {results.box.mr:.4f}")

Ultralytics 8.4.48  Python-3.13.5 torch-2.8.0+cu126 CPU (AMD Ryzen 9 7945HX with Radeon Graphics)
YOLO26n summary (fused): 122 layers, 2,375,811 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access  (ping: 0.40.1 ms, read: 12.50.8 MB/s, size: 71.7 KB)
val: Scanning E:\dev\yjwlYOLO\dev\dataset\val\labels.cache... 14128 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14128/14128 2.5Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 883/883 2.1it/s 7:06<0.5s
                   all      14128     102487      0.764      0.661      0.684      0.548
                   car      14047      87651       0.84      0.808      0.824      0.613
                   bus       2419       2419      0.847       0.75      0.811      0.701
                   van       7296      11750      0.729      0.659      0.686      0.539
                others        666        667      0.641      0.429      0.417      0.339
Speed: 0.4ms prep